# Retrievel Pipeline

### Imports

In [23]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from dotenv import load_dotenv
import os
load_dotenv("D:\Projects\Agentic-AI\.env")

<>:7: SyntaxWarning: invalid escape sequence '\P'
<>:7: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Abdullah\AppData\Local\Temp\ipykernel_1548\203012674.py:7: SyntaxWarning: invalid escape sequence '\P'
  load_dotenv("D:\Projects\Agentic-AI\.env")


True

### Loading Embedings from DB

In [2]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    show_progress=True
)

C:\Users\Abdullah\AppData\Local\Temp\ipykernel_1548\2918223649.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

In [3]:
persist_directory = "db/chroma_db"
vector_store = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings_model,
    collection_metadata={"hnsw:space": "cosine"}
)

### Documents Retriver

In [6]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [7]:
query = "Which island does spaceX lease for its launching in the Pacific?"

retriever_results = retriever.invoke(query)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
retriever_results

[Document(id='e02bd508-4320-4633-b323-16fbbb704d95', metadata={'source': 'D:\\Projects\\Agentic-AI\\langchain-RAG\\docs\\SpaceX.txt'}, page_content="\ufeffSpaceX\nSpace Exploration Technologies Corp., commonly Space Exploration Technologies\nreferred to as SpaceX, is an American space Corp.\ntechnology company headquartered at the Starbase\ndevelopment site in Starbase, Texas.[10] Since its\nfounding in 2002, the company has made numerous\nadvances in rocket propulsion, reusable launch\nvehicles, human spaceflight and satellite\nconstellation technology. As of 2025, SpaceX is the\nworld's dominant space launch provider, its launch\ncadence eclipsing all others, including private\ncompetitors and national programs like the Chinese\nspace program.[11] SpaceX, NASA, and the United\nStates Armed Forces work closely together by\nmeans of governmental contracts.[12]\n\nSpaceX was founded by Elon Musk in 2002 with a Company headquarters, SpaceX Starbase in"),
 Document(id='23e6ef41-73ae-4365-

d:\Projects\Agentic-AI\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Abdullah\.cache\huggingface\hub\models--BAAI--bge-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [18]:
retriever_2 = vector_store.as_retriever(search_type="similarity_score_threshold", 
                                      search_kwargs={"k": 5,
                                                     "score_threshold": 0.5})

retriever_results_2 = retriever_2.invoke(query)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
retriever_results_2

[Document(id='e02bd508-4320-4633-b323-16fbbb704d95', metadata={'source': 'D:\\Projects\\Agentic-AI\\langchain-RAG\\docs\\SpaceX.txt'}, page_content="\ufeffSpaceX\nSpace Exploration Technologies Corp., commonly Space Exploration Technologies\nreferred to as SpaceX, is an American space Corp.\ntechnology company headquartered at the Starbase\ndevelopment site in Starbase, Texas.[10] Since its\nfounding in 2002, the company has made numerous\nadvances in rocket propulsion, reusable launch\nvehicles, human spaceflight and satellite\nconstellation technology. As of 2025, SpaceX is the\nworld's dominant space launch provider, its launch\ncadence eclipsing all others, including private\ncompetitors and national programs like the Chinese\nspace program.[11] SpaceX, NASA, and the United\nStates Armed Forces work closely together by\nmeans of governmental contracts.[12]\n\nSpaceX was founded by Elon Musk in 2002 with a Company headquarters, SpaceX Starbase in"),
 Document(id='23e6ef41-73ae-4365-

### RAG (Model Integration)

In [25]:
query = "Which island does spaceX lease for its launching in the Pacific?"

retriever_2 = vector_store.as_retriever(search_type="similarity_score_threshold", 
                                      search_kwargs={"k": 5,
                                                     "score_threshold": 0.5})

retriever_results_2 = retriever_2.invoke(query)

model = ChatGroq(model="openai/gpt-oss-120b")

combined_input = f"""Based on the following documents, please answer this question: {query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in retriever_results_2])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find the answer in the documents, say "I don't have enough information to answer that question based on the provided documents."
"""

# Define the messages for the model
messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input),
]

# Invoke the model with the combined input
result = model.invoke(messages)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from pprint import pprint
pprint

('SpaceX’s Pacific launch site was **Omelek Island** in the Marshall Islands, '
 'where it conducted its Falcon\u202f1 launches from the Ronald Reagan '
 'Ballistic Missile Defense Test Site.')


In [31]:
print(result.content)

SpaceX’s Pacific launch site was **Omelek Island** in the Marshall Islands, where it conducted its Falcon 1 launches from the Ronald Reagan Ballistic Missile Defense Test Site.


In [32]:
from IPython.display import Markdown, display

display(Markdown(result.content))

SpaceX’s Pacific launch site was **Omelek Island** in the Marshall Islands, where it conducted its Falcon 1 launches from the Ronald Reagan Ballistic Missile Defense Test Site.